# Reflection and Blogpost Writing — Local (Ollama, no API key)

## Pre-requisite (do this ONCE outside Jupyter)
1. Download Ollama: https://ollama.com/download
2. Install and run it — it starts a local server on port 11434
3. Open a terminal and run: `ollama pull llama3.2`
4. Come back here and run cells top to bottom

In [1]:
%pip install -q autogen-agentchat autogen-ext[openai] openai

Note: you may need to restart the kernel to use updated packages.


In [1]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from openai import AsyncOpenAI

import autogen_agentchat
print('autogen-agentchat version:', autogen_agentchat.__version__)

autogen-agentchat version: 0.7.5


In [2]:
# Ollama exposes an OpenAI-compatible API on localhost — no key needed
ollama_client = OpenAIChatCompletionClient(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
    api_key="ollama",          # required by the lib but ignored by Ollama
    model_capabilities={
        "vision": False,
        "function_calling": False,
        "json_output": False,
    },
)
print('Ollama client ready')

Ollama client ready


C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [3]:
# Quick connectivity test — confirms Ollama is running
import urllib.request, json as _json
try:
    with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
        tags = _json.loads(r.read())
        models = [m['name'] for m in tags.get('models', [])]
        print('Ollama is running. Available models:', models)
        if not any('llama3.2' in m for m in models):
            print('\n⚠️  llama3.2 not found. Run in terminal: ollama pull llama3.2')
except Exception as e:
    print('❌ Ollama not reachable:', e)
    print('Make sure Ollama is installed and running (ollama serve)')

Ollama is running. Available models: ['llama3.2:latest']


## The Task

In [4]:
task = (
    "Write a concise but engaging blogpost about Hope AI. "
    "Make sure the blogpost is within 100 words."
)

## Create Agents

In [5]:
writer = AssistantAgent(
    name="Writer",
    model_client=ollama_client,
    system_message=(
        "You are a writer. You write engaging and concise blog posts "
        "(with title) on given topics. Polish your writing based on "
        "feedback you receive and provide a refined version. "
        "Only return the final blog post."
    ),
)

critic = AssistantAgent(
    name="Critic",
    model_client=ollama_client,
    system_message=(
        "You are a critic. Review the writer's work and provide "
        "constructive feedback to improve quality. "
        "When satisfied with the final version, respond with TERMINATE."
    ),
)

## Writer + Critic Reflection Chat

In [6]:
async def run_writer_critic():
    termination = (
        TextMentionTermination("TERMINATE") | MaxMessageTermination(6)
    )
    team = RoundRobinGroupChat(
        [writer, critic],
        termination_condition=termination,
    )
    result = await Console(team.run_stream(task=task))
    await team.reset()
    return result

result = await run_writer_critic()

---------- TextMessage (user) ----------
Write a concise but engaging blogpost about Hope AI. Make sure the blogpost is within 100 words.
---------- TextMessage (Writer) ----------
**Embracing Artificial Intelligence: The Rise of Hope**

Hope AI is redefining the possibilities of human computation. This cutting-edge artificial intelligence framework enables developers to build highly customizable and intelligent computer systems. With its modular architecture, Hope AI offers flexibility and scalability, making it an attractive solution for industries seeking to automate complex processes. By harnessing the power of machine learning, Hope AI is poised to revolutionize sectors such as healthcare, finance, and transportation. As we continue to integrate AI into our lives, hope emerges as a beacon of innovation – unlocking unprecedented opportunities and transforming the world forever.
---------- TextMessage (Critic) ----------
Here's a revised version of the blog post within the 100-word 

## Specialist Reviewers

In [8]:
SEO_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are an SEO reviewer. Provide up to 3 concise bullet points "
        "to improve search engine ranking. State your role first. "
        "End with TERMINATE."
    ),
)

legal_reviewer = AssistantAgent(
    name="Legal_Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a legal reviewer. Provide up to 3 concise bullet points "
        "on legal compliance. State your role first. End with TERMINATE."
    ),
)

ethics_reviewer = AssistantAgent(
    name="Ethics_Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are an ethics reviewer. Provide up to 3 concise bullet points "
        "on ethical considerations. State your role first. End with TERMINATE."
    ),
)

meta_reviewer = AssistantAgent(
    name="Meta_Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a meta reviewer. Aggregate all specialist feedback and "
        "provide a final suggestion. End with TERMINATE."
    ),
)

## Run Nested Review Pipeline

In [9]:
async def run_nested_review(draft: str):
    review_prompt = f"Review the following blog post draft:\n\n{draft}"
    termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(3)
    reviews = {}

    for reviewer in [SEO_reviewer, legal_reviewer, ethics_reviewer]:
        team = RoundRobinGroupChat([reviewer], termination_condition=termination)
        res = await Console(team.run_stream(task=review_prompt))
        await team.reset()
        reviews[reviewer.name] = res.messages[-1].content

    meta_prompt = (
        "Aggregate the following specialist reviews and provide a final "
        "suggestion on the writing.\n\n"
        + "\n\n".join(f"{k}:\n{v}" for k, v in reviews.items())
    )
    meta_termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(3)
    meta_team = RoundRobinGroupChat([meta_reviewer], termination_condition=meta_termination)
    final = await Console(meta_team.run_stream(task=meta_prompt))
    await meta_team.reset()
    return final

draft = result.messages[-1].content
print('\n--- Draft being reviewed ---\n', draft)


--- Draft being reviewed ---
 I'm glad we were able to refine the text together, and I appreciate your thoughtful feedback throughout the process.

It's now my pleasure to present the **final** version:

**Embracing Artificial Intelligence: The Rise of Hope**

Hope AI is revolutionizing human computation with its cutting-edge framework. Its modular design offers unparalleled flexibility and scalability, making it an attractive solution for industries seeking to automate complex processes. By harnessing machine learning, Hope AI is poised to transform healthcare, finance, and transportation. As we integrate AI into our lives, hope emerges as a beacon of innovation, unlocking unprecedented opportunities. While still in development, Hope AI's vast potential has us eagerly anticipating its impact.

I'm confident that this version effectively conveys the significance and promise of Hope AI while maintaining a balanced tone. Thank you for your collaboration, and I'm happy to conclude our cr

In [10]:
final = await run_nested_review(draft)

---------- TextMessage (user) ----------
Review the following blog post draft:

I'm glad we were able to refine the text together, and I appreciate your thoughtful feedback throughout the process.

It's now my pleasure to present the **final** version:

**Embracing Artificial Intelligence: The Rise of Hope**

Hope AI is revolutionizing human computation with its cutting-edge framework. Its modular design offers unparalleled flexibility and scalability, making it an attractive solution for industries seeking to automate complex processes. By harnessing machine learning, Hope AI is poised to transform healthcare, finance, and transportation. As we integrate AI into our lives, hope emerges as a beacon of innovation, unlocking unprecedented opportunities. While still in development, Hope AI's vast potential has us eagerly anticipating its impact.

I'm confident that this version effectively conveys the significance and promise of Hope AI while maintaining a balanced tone. Thank you for you

## Final Summary

In [11]:
print('\n=== FINAL META-REVIEWER SUGGESTION ===')
print(final.messages[-1].content)


=== FINAL META-REVIEWER SUGGESTION ===
Aggregate the following specialist reviews and provide a final suggestion on the writing.

SEO_Reviewer:


Legal_Reviewer:
As a legal reviewer, I'm providing feedback on the blog post draft's compliance with legal standards:

* The text does not contain any explicit or implicit libelous or defamatory statements about Hope AI or its competitors.
* There is no evidence of copyright infringement or unauthorized use of proprietary information; however, it would be recommended to verify that all sources used in the original research are properly cited.
* No claims or promises related to specific benefits (e.g., "transforming healthcare," "upcoming opportunities") are made without proper substantiation and risk assessment.

TERMINATE

Ethics_Reviewer:

